# Question 11 — Sharpe ratio maximization

Model (from the handwritten notes):

* $N = 80$ applicants; each picks 5 of the 11 questions, so baseline $p = 5/11$.
* $A(i) \sim \text{Bin}(80, p_i)$ with $p_i = 5/11 - 0.15$ (Unpopular), $5/11$ (Moderate), $5/11 + 0.15$ (Popular).
* Guess $m_i$ for question $i$: score $S(i) = +2^{m_i}$ if $A(i) \ge m_i$, else $-2^{m_i}$.
* $T = \sum_i S(i)$, questions independent. Maximize $\varphi = E[T] / (\sqrt{\text{Var}(T)} + c)$ over $m_i \in \{1, \dots, 80\}$ using Nelder–Mead.

In [1]:
import numpy as np
from scipy.stats import binom
from scipy.optimize import minimize

N = 80
BASE = 5 / 11
DELTA = 0.15
C = 1  # constant added to the Sharpe denominator
USE_DENOMINATOR = True  # False: drop the denominator and maximize E[T] only
POPULARITY_P = {"U": BASE - DELTA, "M": BASE, "P": BASE + DELTA}

# Q1 .. Q11
labels = ["U", "U", "P", "M", "P", "U", "P", "P", "U", "M", "P"]
p = np.array([POPULARITY_P[l] for l in labels])
p

array([0.30454545, 0.30454545, 0.60454545, 0.45454545, 0.60454545,
       0.30454545, 0.60454545, 0.60454545, 0.30454545, 0.45454545,
       0.60454545])

## Closed forms for $E[S(i)]$ and $\text{Var}(S(i))$

$F(i) = P(A(i) \ge m_i) = \sum_{j=m_i}^{80} \binom{80}{j} p_i^j (1-p_i)^{80-j}$, and $D(i) = P(A(i) < m_i) = 1 - F(i)$.

$$E[S(i)] = 2^{m_i}\,[F(i) - D(i)]$$

Because $S(i) = \pm 2^{m_i}$, $E[S(i)^2] = 2^{2m_i}$, so

$$\text{Var}(S(i)) = 2^{2m_i}\left(1 - (F - D)^2\right) = 2^{2m_i} \cdot 4\,F(i)\,D(i).$$

(The notes give $(2^{2m_i} - 2^{m_i})(F - D)$, which is not the variance: it is negative whenever $F < D$.)

$F$ and $D$ come from `binom.sf` and `binom.cdf` separately, not from `1 - F`. When $F \approx 1$, `1 - F` cancels to 0 in floating point and the variance falsely becomes 0.

In [2]:
def F(m, p):
    """P(A >= m)"""
    return binom.sf(m - 1, N, p)


def D(m, p):
    """P(A < m)"""
    return binom.cdf(m - 1, N, p)


def question_moments(m, p):
    f, d = F(m, p), D(m, p)
    scale = 2.0 ** m
    mean = scale * (f - d)
    var = scale**2 * 4 * f * d
    return mean, var


def sharpe(m, c=C, p=p):
    mean, var = question_moments(np.asarray(m), p)
    if not USE_DENOMINATOR:
        return mean.sum()
    return mean.sum() / (np.sqrt(var.sum()) + c)

## Why the constant $c$

With $\varphi = E[T]/\sqrt{\text{Var}(T)}$, near-certain bets push the variance to about 0 and $\varphi$ to infinity. Adding $c$ caps the reward for low risk: once $\sqrt{\text{Var}(T)} \ll c$, $\varphi \approx E[T]/c$, so the optimizer gains by raising $m_i$ (bigger payoffs) until the added risk becomes comparable to $c$. A larger $c$ tolerates more risk.

Setting `USE_DENOMINATOR = False` drops the denominator entirely, so `sharpe` returns $E[T]$ and the optimization becomes pure expected-reward maximization (no risk penalty).

## Nelder–Mead

Nelder–Mead works on continuous inputs, so the objective rounds and clips $x$ to integers in $[1, 80]$ before evaluating. That makes the objective piecewise constant, so we use a wide initial simplex (step 10) and many random restarts, then keep the best result.

In [3]:
def objective(x, c):
    m = np.clip(np.rint(x), 1, N)
    return -sharpe(m, c)


def nelder_mead(c=C, n_restarts=50, step=10.0, seed=0):
    rng = np.random.default_rng(seed)
    k = len(p)
    best = None
    for _ in range(n_restarts):
        x0 = rng.uniform(1, N, k)
        simplex = np.vstack([x0] + [x0 + step * e for e in np.eye(k)])
        res = minimize(
            objective,
            x0,
            args=(c,),
            method="Nelder-Mead",
            options={"initial_simplex": simplex, "maxfev": 20000, "xatol": 0.5, "fatol": 1e-12},
        )
        if best is None or res.fun < best.fun:
            best = res
    return np.clip(np.rint(best.x), 1, N).astype(int), -best.fun


m_nm, phi_nm = nelder_mead()
print("Nelder-Mead m* =", m_nm, " objective =", phi_nm)

Nelder-Mead m* = [ 1  1  1  1 18  1 18  1  1  1 15]  objective = 321556.59268950054


Rounding makes the objective flat over large regions (moving an $m_i$ that sits at 1 changes $\varphi$ only slightly), so Nelder–Mead tends to stop early. We polish its result with integer coordinate ascent: for each question in turn, try every $m_i \in \{1, \dots, 80\}$ with the others held fixed, keep any improvement, and repeat until nothing changes.

In [4]:
def coordinate_ascent(m, c=C):
    m = m.copy()
    best = sharpe(m, c)
    improved = True
    while improved:
        improved = False
        for i in range(len(m)):
            for k in range(1, N + 1):
                trial = m.copy()
                trial[i] = k
                val = sharpe(trial, c)
                if val > best + 1e-12:
                    m, best, improved = trial, val, True
    return m, best


m_opt, phi_opt = coordinate_ascent(m_nm)
print("Polished m* =", m_opt, " objective =", phi_opt)

Polished m* = [ 4  4 18 10 18  4 18 18  4 10 18]  objective = 608295.5944269908


In [5]:
mean, var = question_moments(m_opt, p)
print(f"{'Q':>3} {'label':>5} {'p':>6} {'m':>3} {'P(A>=m)':>10} {'E[S]':>12} {'sd(S)':>12}")
for i in range(len(p)):
    print(f"{i+1:>3} {labels[i]:>5} {p[i]:>6.3f} {m_opt[i]:>3} {F(m_opt[i], p[i]):>10.4g} {mean[i]:>12.4g} {np.sqrt(var[i]):>12.4g}")
print(f"\nE[T] = {mean.sum():.4g},  sd(T) = {np.sqrt(var.sum()):.4g},  objective = {sharpe(m_opt):.4g}")

  Q label      p   m    P(A>=m)         E[S]        sd(S)
  1     U  0.305   4          1           16     0.001363
  2     U  0.305   4          1           16     0.001363
  3     P  0.605  18          1    2.621e+05       0.5179
  4     M  0.455  10          1         1024      0.01388
  5     P  0.605  18          1    2.621e+05       0.5179
  6     U  0.305   4          1           16     0.001363
  7     P  0.605  18          1    2.621e+05       0.5179
  8     P  0.605  18          1    2.621e+05       0.5179
  9     U  0.305   4          1           16     0.001363
 10     M  0.455  10          1         1024      0.01388
 11     P  0.605  18          1    2.621e+05       0.5179

E[T] = 1.313e+06,  sd(T) = 1.158,  objective = 6.083e+05


Check: without the denominator the objective is separable, $E[T] = \sum_i E[S(i)]$, so the exact optimum is each question's own $\arg\max_{m_i} E[S(i)]$. A brute-force search over $m_i \in \{1, \dots, 80\}$ per question confirms the optimizer.

In [6]:
if not USE_DENOMINATOR:
    grid = np.arange(1, N + 1)
    m_exact = np.array([grid[np.argmax(question_moments(grid, pi)[0])] for pi in p])
    print("Exact m* =", m_exact, " E[T] =", sharpe(m_exact))
    print("Optimizer matches exact:", np.array_equal(m_exact, m_opt))

## Sensitivity to $c$ (only runs when `USE_DENOMINATOR = True`)

In [7]:
for c in ([1, 10, 100, 1000] if USE_DENOMINATOR else []):
    m_c, _ = coordinate_ascent(nelder_mead(c)[0], c)
    mean_c, var_c = question_moments(m_c, p)
    print(f"c = {c:>4}: m* = {m_c}  Sharpe = {sharpe(m_c, c):.4g}  E[T] = {mean_c.sum():.4g}  sd(T) = {np.sqrt(var_c.sum()):.4g}")

c =    1: m* = [ 4  4 18 10 18  4 18 18  4 10 18]  Sharpe = 6.083e+05  E[T] = 1.313e+06  sd(T) = 1.158


c =   10: m* = [ 6  6 20 12 20  6 20 20  6 12 20]  Sharpe = 1.53e+05  E[T] = 5.251e+06  sd(T) = 24.32


c =  100: m* = [ 7  7 21 14 21  7 21 21  7 14 21]  Sharpe = 5.1e+04  E[T] = 1.052e+07  sd(T) = 106.3


c = 1000: m* = [ 9  9 23 15 23  9 23 23  9 15 23]  Sharpe = 1.486e+04  E[T] = 4.201e+07  sd(T) = 1826
